##**Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Gold
### Objetivo
Construir a **tabela fato de vendas** do Squad 2, enriquecendo os itens de pedido validados com dados de pedidos (via JOIN) e métricas calculadas de receita e desconto, disponibilizando o resultado para o SQL Server.
### Origem e Destinos
| Item | Valor |
| **Origem principal** | `squad2/silver/ecommerce_itens_pedido` (Delta Lake) |
| **Origem lookup** | `squad2/silver/ecommerce_pedidos` (Delta Lake) |
| **Destino Lake** | `squad2/gold/ecommerce_itens_pedido` (Delta Lake — append) |
| **Destino SQL** | `squad2.gold_ecommerce_itens_pedido` (SQL Server — append) |
| **Controle** | `gold/control/ecommerce_itens_pedido.json` |
### Regras de Negócio Aplicadas
| # | Regra | Descrição | Saída |
| 6 | SKUs únicos vendidos na última hora | Contagem de produtos distintos no período | Métrica de monitoramento |
| 7 | Top 5 SKUs em 30 minutos | Ranking dos mais vendidos no período | Métrica de monitoramento |
| 8 | Receita Bruta vs Líquida | Alerta se desconto > 25% da receita bruta | Log de alerta |
| 9 | Desconto impossível | `desconto_aplicado > preco_unitario` | Campo `alerta_desconto_impossivel` |
| 10 | Média de itens por pedido | Alerta se média < 3 itens/pedido | Log de alerta |
### Campos Calculados na Gold
| Campo | Lógica | Tipo |
| `receita_bruta_calculada` | `quantidade × preco_unitario` | float |
| `receita_liquida_calculada` | `receita_bruta_calculada − desconto_aplicado` | float |
| `alerta_desconto_impossivel` | `desconto_aplicado > preco_unitario` | `'SIM'`/`'NAO'` |
| `gold_processed_at` | Timestamp de processamento na Gold | datetime |
| `data_pedido` | Herdado via JOIN com `ecommerce_pedidos` | date |
| `id_cliente` | Herdado via JOIN com `ecommerce_pedidos` | string |
| `status` | Herdado via JOIN com `ecommerce_pedidos` | string |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | `get_storage_options`, `get_squad2_client`, `SQL_OPTIONS` |
| `feat_squad2_silver_itens_pedido` | Dados validados de itens |
| Silver `ecommerce_pedidos` | Lookup de dados do pedido pai |


In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA_ITENS = "ecommerce_itens_pedido"
TABELA_PEDIDOS = "ecommerce_pedidos"
TABELA_SQL = f"gold_{TABELA_ITENS}"  

path_silver_itens   = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA_ITENS}"
path_silver_pedidos = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA_PEDIDOS}"
path_gold           = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA_ITENS}"
path_control        = f"control/gold/{TABELA_ITENS}/control_file.json"

- Construção da Tabela Fato e Gravação

**Pré-condições verificadas:**
- Silver de itens deve estar inicializada
- Silver de pedidos deve estar inicializada (necessária para o JOIN)
**JOIN com Pedidos:** enriquece cada item com `data_pedido`, `id_cliente` e `status` do pedido pai.
O JOIN é `left` para não descartar itens cujo pedido não seja encontrado.
 **Métricas calculadas por linha:**
- `receita_bruta_calculada = quantidade × preco_unitario`
- `receita_liquida_calculada = receita_bruta - desconto_aplicado`
- `alerta_desconto_impossivel = 'SIM'` quando desconto supera o preço unitário
**Sink duplo:** dados gravados simultaneamente no Delta Lake e no SQL Server.

In [0]:
try:
    if not DeltaTable.is_deltatable(path_silver_itens, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de Itens {TABELA_ITENS} ainda não foi inicializada.")
    elif not DeltaTable.is_deltatable(path_silver_pedidos, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de Pedidos {TABELA_PEDIDOS} não foi localizada.")
    else:
        dt_itens        = DeltaTable(path_silver_itens, storage_options=get_storage_options())
        df_itens_pandas = dt_itens.to_pandas()
 
        squad2_client = get_squad2_client()
        file_client   = squad2_client.get_file_client(path_control)
 
        processados = set()
        if file_client.exists():
            conteudo    = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
 
        df_itens_novos = df_itens_pandas[~df_itens_pandas['bronze_source_file'].isin(processados)].copy()
 
        if df_itens_novos.empty:
            print(" Camada Gold de Itens de Pedido em dia! Nenhum registro novo para processar.")
        else:
            print(f"  Processando {len(df_itens_novos)} linhas para a Fato...")
 
            # JOIN LEFT com Silver de Pedidos para enriquecer cada item com dados do pedido pai
            dt_pedidos        = DeltaTable(path_silver_pedidos, storage_options=get_storage_options())
            df_pedidos_pandas = dt_pedidos.to_pandas()
 
            colunas_pedidos   = ['id_pedido', 'data_pedido', 'id_cliente', 'status']
            colunas_existente = [c for c in colunas_pedidos if c in df_pedidos_pandas.columns]
            df_pedidos_lookup = df_pedidos_pandas[colunas_existente].drop_duplicates(subset=['id_pedido'])
 
            df_gold_final = pd.merge(df_itens_novos, df_pedidos_lookup, on='id_pedido', how='left')
 
            # Regra N9: campo alerta por linha — desconto impossível (desconto > preço unitário)
            df_gold_final['receita_bruta_calculada']    = df_gold_final['quantidade'] * df_gold_final['preco_unitario']
            df_gold_final['receita_liquida_calculada']  = df_gold_final['receita_bruta_calculada'] - df_gold_final['desconto_aplicado']
            df_gold_final['alerta_desconto_impossivel'] = df_gold_final.apply(
                lambda row: 'SIM' if row['desconto_aplicado'] > row['preco_unitario'] else 'NAO', axis=1
            )
 
            df_gold_final['gold_processed_at'] = datetime.now()
 
            # Remove timezone de colunas datetime (requisito do formato Delta)
            for col in df_gold_final.columns:
                if pd.api.types.is_datetime64_any_dtype(df_gold_final[col]):
                    df_gold_final[col] = df_gold_final[col].dt.tz_localize(None)
 
            # SINK 1: Delta Lake
            write_deltalake(path_gold, df_gold_final, mode="append", storage_options=get_storage_options())
 
            # SINK 2: SQL Server
            try:
                df_schema_sql = spark.read \
                    .format("sqlserver") \
                    .options(**SQL_OPTIONS) \
                    .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                    .load() \
                    .limit(0)
 
                spark_df_final = spark.createDataFrame(df_gold_final)
                for col_db in df_schema_sql.columns:
                    col_match = [c for c in spark_df_final.columns if c.lower() == col_db.lower()]
                    if col_match:
                        spark_df_final = spark_df_final.withColumnRenamed(col_match[0], col_db)
                spark_df_aligned = spark_df_final.select(*df_schema_sql.columns)
                print(f"  Tabela existente localizada. Alinhando colunas e fazendo Append...")
            except Exception:
                print(f"  Criando nova tabela Fato: [squad2].[{TABELA_SQL}]...")
                spark_df_aligned = spark.createDataFrame(df_gold_final)
 
            spark_df_aligned.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                .mode("append") \
                .save()
 
            arquivos_atuais   = set(df_itens_novos['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
 
            print(f"\n SUCESSO! Itens enriquecidos salvos em squad2.{TABELA_SQL}!")
 
except Exception as e:
    print(f"  Erro no processamento: {str(e)}")
    raise

### 4. KPIs e Alertas de Negócio em Tempo Real
Esta seção lê a Gold já consolidada e calcula os KPIs sobre janelas temporais.
Os alertas são emitidos como logs — podem ser integrados a notificações (Teams, e-mail) conforme necessário.

| # | KPI / Alerta | Janela | Gatilho |

| N6 | SKUs únicos vendidos | Última 1 hora | Monitoramento de diversidade |

| N7 | Top 5 SKUs por quantidade | Últimos 30 minutos | Concentração anormal / promoção viral |

| N8 | Impacto dos descontos na receita | Lote atual | Desconto > 25% da receita bruta → alerta |

| N10 | Média de itens por pedido | Última 1 hora | Média < 3 → possível problema no checkout |

In [0]:
try:
    from datetime import datetime, timedelta
    if not DeltaTable.is_deltatable(path_gold, storage_options=get_storage_options()):
        print(" [KPIs] Gold ainda não inicializada — KPIs serão calculados na próxima execução.")
    else:
        dt_gold   = DeltaTable(path_gold, storage_options=get_storage_options())
        df_kpi    = dt_gold.to_pandas()
        agora     = datetime.now()
        janela_1h = agora - timedelta(hours=1)
        janela_30 = agora - timedelta(minutes=30)
 
        # Garante que gold_processed_at é datetime para filtros temporais
        df_kpi['gold_processed_at'] = pd.to_datetime(df_kpi['gold_processed_at'])
 
        df_1h = df_kpi[df_kpi['gold_processed_at'] >= janela_1h]
        df_30 = df_kpi[df_kpi['gold_processed_at'] >= janela_30]
 
        # ── Regra N6: SKUs únicos na última hora ──────────────────────────────
        # Mede a diversidade do catálogo sendo vendido. Queda brusca indica
        # problema no catálogo, no buscador ou concentração anormal de demanda.
        skus_unicos_1h = df_1h['sku'].nunique()
        print(f"\n[N6] SKUs únicos vendidos na última hora: {skus_unicos_1h}")
 
        # ── Regra N7: Top 5 SKUs mais vendidos nos últimos 30 minutos ─────────
        # Concentração excessiva em 1 SKU pode indicar bot, scraper ou promoção viral.
        if not df_30.empty:
            top5 = (
                df_30.groupby('sku')['quantidade']
                .sum()
                .sort_values(ascending=False)
                .head(5)
                .reset_index()
            )
            print(f"\n[N7] Top 5 SKUs mais vendidos (últimos 30 min):")
            for _, row in top5.iterrows():
                print(f"     SKU {row['sku']}: {int(row['quantidade'])} unidades")
        else:
            print("\n[N7] Sem dados nos últimos 30 minutos para calcular Top 5.")
 
        # ── Regra N8: Alerta de impacto de desconto na receita bruta ──────────
        # Desconto > 25% da receita bruta do lote indica política agressiva ou erro.
        if not df_kpi.empty and 'receita_bruta_calculada' in df_kpi.columns:
            receita_bruta  = df_kpi['receita_bruta_calculada'].sum()
            receita_liq    = df_kpi['receita_liquida_calculada'].sum()
            total_desconto = receita_bruta - receita_liq
            pct_desconto   = (total_desconto / receita_bruta * 100) if receita_bruta > 0 else 0
 
            print(f"\n[N8] Receita Bruta total  : R$ {receita_bruta:,.2f}")
            print(f"[N8] Receita Líquida total: R$ {receita_liq:,.2f}")
            print(f"[N8] Desconto total       : R$ {total_desconto:,.2f} ({pct_desconto:.1f}%)")
 
            if pct_desconto > 25:
                print(f"\n  [ALERTA N8] Desconto representa {pct_desconto:.1f}% da receita bruta — acima do limite de 25%!")
 
        # ── Regra N10: Média de itens por pedido na última hora ───────────────
        # Queda abrupta abaixo de 3 itens/pedido pode indicar problema no checkout
        # (carrinho sendo esvaziado antes de fechar).
        if not df_1h.empty:
            media_itens = df_1h.groupby('id_pedido')['id_item_pedido'].count().mean()
            print(f"\n[N10] Média de itens por pedido (última hora): {media_itens:.2f}")
 
            if media_itens < 3:
                print(f"  [ALERTA N10] Média de {media_itens:.2f} itens/pedido abaixo do mínimo esperado (3).")
                print(f"    Verificar fluxo de checkout — possível abandono de carrinho.")
        else:
            print("\n[N10] Sem dados na última hora para calcular média de itens por pedido.")
 
except Exception as e:
    print(f" [KPIs] Erro ao calcular KPIs: {str(e)}")